# 03 — Normalization and Sliding Windows

This notebook converts engineered OHLCV features into model-ready tensors:
1. split data by time
2. normalize using train-only statistics
3. build rolling windows for sequence models

In [ ]:
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

In [ ]:
path = Path("../../data/ohlcv_with_indicators.csv")
if path.exists():
    df = pd.read_csv(path, index_col=0, parse_dates=True)
else:
    raise FileNotFoundError(
        "Missing ../../data/ohlcv_with_indicators.csv. Run 02_technical_indicators.ipynb first."
    )

df = df.dropna().copy()
print("Loaded rows:", len(df))
print("Columns:", list(df.columns))

## 1) Define target

We create a binary label: `1` if next-day close is higher than today, else `0`.

In [ ]:
df["target_up"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
df = df.iloc[:-1].copy()  # remove last row without next-day target

feature_cols = [c for c in df.columns if c != "target_up"]
X_all = df[feature_cols].values
y_all = df["target_up"].values

print("Feature matrix shape:", X_all.shape)
print("Target shape:", y_all.shape)
print("Target mean (class-1 ratio):", y_all.mean().round(3))

## 2) Time-based split (no shuffling)

In [ ]:
n = len(df)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

X_train = X_all[:n_train]
y_train = y_all[:n_train]

X_val = X_all[n_train:n_train+n_val]
y_val = y_all[n_train:n_train+n_val]

X_test = X_all[n_train+n_val:]
y_test = y_all[n_train+n_val:]

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

## 3) Normalize with MinMaxScaler

Fit only on train set to avoid leakage.

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train scaled min/max:", X_train_scaled.min().round(3), X_train_scaled.max().round(3))
print("Val scaled min/max:  ", X_val_scaled.min().round(3), X_val_scaled.max().round(3))

## 4) Build sliding windows

For each sample, use the previous `window_size` rows as input sequence.

In [ ]:
def make_windows(X, y, window_size=30):
    X_seq, y_seq = [], []
    for i in range(window_size, len(X)):
        X_seq.append(X[i-window_size:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

window_size = 30
X_train_seq, y_train_seq = make_windows(X_train_scaled, y_train, window_size)
X_val_seq, y_val_seq = make_windows(X_val_scaled, y_val, window_size)
X_test_seq, y_test_seq = make_windows(X_test_scaled, y_test, window_size)

print("X_train_seq:", X_train_seq.shape)
print("X_val_seq:  ", X_val_seq.shape)
print("X_test_seq: ", X_test_seq.shape)

In [ ]:
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_t = torch.tensor(y_train_seq, dtype=torch.long)
X_val_t = torch.tensor(X_val_seq, dtype=torch.float32)
y_val_t = torch.tensor(y_val_seq, dtype=torch.long)
X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_t = torch.tensor(y_test_seq, dtype=torch.long)

print("Torch tensors ready ✅")
print("Example input sequence shape:", X_train_t[0].shape)  # [window_size, n_features]

## 5) Save tensors for next modules

In [ ]:
out = Path("../../data/time_series_tensors.npz")
np.savez(
    out,
    X_train=X_train_seq, y_train=y_train_seq,
    X_val=X_val_seq, y_val=y_val_seq,
    X_test=X_test_seq, y_test=y_test_seq,
    feature_cols=np.array(feature_cols, dtype=object),
    window_size=window_size,
)
print("Saved:", out.resolve())

## 6) Exercises

1. Change `window_size` to 60 and compare resulting sample count.
2. Create a 3-class target (`down`, `flat`, `up`) using return thresholds.
3. Replace MinMax scaling with StandardScaler and compare distributions.

You now have model-ready time-series sequences for LSTM/Transformer modules.